# Sentiment Analysis con Transformer en PyTorch
Este notebook muestra un caso de uso práctico usando **PyTorch** y un modelo tipo **Transformer**.

El objetivo: clasificar texto como **positivo** o **negativo**.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import random
import numpy as np


## 1. Dataset de ejemplo

In [2]:
data = [
    ("I love this movie, it was fantastic!", 1),
    ("Absolutely terrible, waste of time.", 0),
    ("What a great experience, I enjoyed it a lot", 1),
    ("Horrible acting, very bad plot.", 0),
    ("A masterpiece, brilliant work!", 1),
    ("Worst film ever, do not recommend.", 0)
]

# Diccionario simple
vocab = {}
for text, _ in data:
    for word in text.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab) + 1
vocab_size = len(vocab) + 1
print("Vocabulario:", vocab)


Vocabulario: {'i': 1, 'love': 2, 'this': 3, 'movie,': 4, 'it': 5, 'was': 6, 'fantastic!': 7, 'absolutely': 8, 'terrible,': 9, 'waste': 10, 'of': 11, 'time.': 12, 'what': 13, 'a': 14, 'great': 15, 'experience,': 16, 'enjoyed': 17, 'lot': 18, 'horrible': 19, 'acting,': 20, 'very': 21, 'bad': 22, 'plot.': 23, 'masterpiece,': 24, 'brilliant': 25, 'work!': 26, 'worst': 27, 'film': 28, 'ever,': 29, 'do': 30, 'not': 31, 'recommend.': 32}


In [3]:
class TextDataset(Dataset):
    def __init__(self, data, vocab, max_len=10):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        tokens = [self.vocab.get(word, 0) for word in text.lower().split()]
        if len(tokens) < self.max_len:
            tokens += [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        return torch.tensor(tokens), torch.tensor(label)

dataset = TextDataset(data, vocab)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
for X, y in loader:
    print(X, y)
    break


tensor([[19, 20, 21, 22, 23,  0,  0,  0,  0,  0],
        [13, 14, 15, 16,  1, 17,  5, 14, 18,  0]]) tensor([0, 1])


## 2. Modelo Transformer en PyTorch

In [4]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2, hidden_dim=64, num_layers=2, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq, embed)
        embedded = embedded.permute(1, 0, 2)  # (seq, batch, embed)
        encoded = self.transformer(embedded)
        pooled = encoded.mean(dim=0)
        return self.fc(pooled)

model = TransformerClassifier(vocab_size)
print(model)


TransformerClassifier(
  (embedding): Embedding(33, 32)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=32, out_features=2, bias=True)
)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


## 3. Entrenamiento

In [5]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.8810
Epoch 2, Loss: 0.6751
Epoch 3, Loss: 0.5813
Epoch 4, Loss: 0.5651
Epoch 5, Loss: 0.6402


## 4. Predicciones

In [6]:
test_text = "I really loved this amazing film"
tokens = [vocab.get(word, 0) for word in test_text.lower().split()]
tokens = tokens + [0]*(10-len(tokens)) if len(tokens)<10 else tokens[:10]
input_tensor = torch.tensor([tokens])
with torch.no_grad():
    output = model(input_tensor)
    prediction = torch.argmax(output, dim=1).item()
print(f"Texto: '{test_text}' -> Sentimiento: {'Positivo' if prediction==1 else 'Negativo'}")


Texto: 'I really loved this amazing film' -> Sentimiento: Negativo
